# Finance AI Assistant - Merged Notebook
This notebook merges Stage 1 (non-instruction fine-tuning), Stage 2 (instruction SFT), and Stage 3 (DPO alignment) into a single runnable flow.
Run all cells top-to-bottom in Colab for end-to-end demo outputs.

# Stage 1 — Finance Non-Instruction Fine-Tuning
This standalone Colab notebook adapts a Qwen2.5 0.5B model to finance and banking language using raw domain text. It includes preprocessing, QLoRA training, evaluation, adapter saving, and a Finance-only ipywidgets demo.

In [ ]:
# Install pinned, Colab-compatible packages
!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib


In [ ]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


In [ ]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


## Load and chunk raw finance text

In [ ]:
raw_text=(DATA_DIR/"non_instruction_data.txt").read_text(encoding="utf-8")
paragraphs=[p.strip() for p in raw_text.split("\n\n") if p.strip()]
chunks=[]
buffer=""
for p in paragraphs:
    if len(buffer)+len(p)<1800: buffer=(buffer+"\n\n"+p).strip()
    else: chunks.append({"text":buffer}); buffer=p
if buffer: chunks.append({"text":buffer})
train_dataset=Dataset.from_list(chunks)
print("Paragraphs:",len(paragraphs),"Chunks:",len(train_dataset))
print(train_dataset[0]["text"][:600])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


## Train Stage 1

In [ ]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR=Path("/content/finance_adapters/stage1_non_instruction")
args=SFTConfig(output_dir=str(OUTPUT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=1,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=train_dataset,args=args)
result=trainer.train()
model.save_pretrained(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved:",OUTPUT_DIR)

## Step 5 - Base model evaluation (before instruction fine-tuning)

In [ ]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

# Step 5 uses these 10 questions for the base-model table.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=220, temperature=0.2):
    prompt = f"Finance domain context question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

def diagnose_problem(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)

    if len(words) < 22:
        return "Answer is too brief and misses important finance-specific detail."

    vague_markers = ["contact your bank", "check with your bank", "it depends", "usually", "generally"]
    if any(marker in a for marker in vague_markers):
        return "Answer is generic and not specific enough to finance workflows or policy context."

    if "otp" in q and not any(k in a for k in ["never share", "fraud", "report", "block", "cybercrime"]):
        return "Safety guidance is incomplete for fraud/OTP risk handling."

    if any(k in q for k in ["how do i", "how can i", "what should i do"]) and not any(
        k in a for k in ["step", "log in", "portal", "app", "documents", "submit", "contact support", "immediately"]
    ):
        return "Procedural guidance is weak; answer does not provide actionable steps."

    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil", "interest", "loan", "deposit", "bank", "account"
    ]
    if not any(term in a for term in domain_terms):
        return "Answer lacks domain terminology and appears too general."

    return "Answer is partially correct but can be more specific, complete, and policy-grounded."

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
problems = {q: diagnose_problem(q, base_answers[q]) for q in EVAL_QUESTIONS}

df_base = pd.DataFrame([
    {"Question": q, "Base Model Answer": base_answers[q], "Problem": problems[q]}
    for q in EVAL_QUESTIONS
])
display(df_base)

# Export Step 5 table to reports/base_model_evaluation.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "base_model_evaluation.md"

lines = [
    "# Base Model Evaluation (Finance FAQ Assistant)",
    "",
    "This report captures baseline behavior of the original base model before instruction fine-tuning.",
    "",
    "## Evaluation Setup",
    "",
    "- Domain: Finance FAQ Assistant",
    "- Dataset reference: `data/instruction_dataset.jsonl`",
    "- Number of evaluation questions: 10",
    "- Goal: Identify generic, incomplete, or non-domain-specific behavior in base model outputs.",
    "",
    "## Baseline Results",
    "",
    "| Question | Base Model Answer | Problem |",
    "| --- | --- | --- |",
]
for q in EVAL_QUESTIONS:
    ans = " ".join(base_answers[q].split()).replace("|", "\\|")
    prob = " ".join(problems[q].split()).replace("|", "\\|")
    lines.append(f"| {q} | {ans} | {prob} |")

lines += [
    "",
    "## Summary",
    "",
    "- Typical baseline gaps observed: Generic language, limited procedural detail, and missing safety specifics in some responses.",
    "- Most common failure type (generic, inaccurate, unsafe, incomplete): Generic and incomplete responses.",
    "- Key areas to improve through SFT: Domain terminology precision, step-by-step instructions, and safer financial guidance.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 5 report written:", out_path)

## Interactive demo

In [ ]:
# Finance-only ipywidgets demo
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


# Stage 2 — Finance Instruction Fine-Tuning (SFT)
This standalone notebook fine-tunes a Finance FAQ Assistant on verified instruction–response examples. It includes base-model evaluation, Alpaca-style prompt formatting, QLoRA SFT, before/after evaluation, adapter saving, and a Finance-only ipywidgets demo.

In [ ]:
# Install pinned, Colab-compatible packages (already handled at the beginning)
#!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib
print("Skipping Stage 2 package install; dependencies already installed in Cell 3.")

In [ ]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


In [ ]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


In [ ]:
rows=[json.loads(x) for x in (DATA_DIR/"instruction_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
print("Instruction examples:",len(rows))
def format_row(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token if 'tokenizer' in globals() else ''}"}


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


In [ ]:
def format_row(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token}"}
train_dataset=Dataset.from_list([format_row(r) for r in rows])
print(train_dataset[0]["text"][:800])

## Step 7 - Base vs SFT comparison (same 10 questions)

In [ ]:
# Capture base answers for Step 7 using the same 10 questions used in Step 5.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=240, temperature=0.2):
    prompt = f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in base_answers.items():
    print("\nQ:", q, "\nBase:", a)

## Train Stage 2 SFT

In [ ]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR=Path("/content/finance_adapters/stage2_sft")
args=SFTConfig(output_dir=str(OUTPUT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=2,learning_rate=2e-4,warmup_ratio=0.05,logging_steps=5,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=train_dataset,args=args)
trainer.train(); model.save_pretrained(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved:",OUTPUT_DIR)

## Step 7 - Comparison table and report export

In [ ]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

def score_answer(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)
    score = 0
    notes = []

    # Length/detail
    if len(words) >= 30:
        score += 2
    elif len(words) >= 18:
        score += 1
    else:
        notes.append("too brief")

    # Domain specificity
    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil",
        "interest", "loan", "deposit", "bank", "account", "ifsc", "otp"
    ]
    domain_hits = sum(1 for t in domain_terms if t in a)
    if domain_hits >= 2:
        score += 2
    elif domain_hits == 1:
        score += 1
    else:
        notes.append("low domain specificity")

    # Safety behavior for fraud/OTP questions
    if "otp" in q or "unauthorized" in q or "suspicious" in q:
        if any(k in a for k in ["never share", "report", "fraud", "block", "cybercrime", "helpline"]):
            score += 2
        else:
            notes.append("missing safety guidance")

    # Procedural clarity for how-to questions
    if any(k in q for k in ["how do i", "how can i", "what should i do"]):
        if any(k in a for k in ["log in", "visit", "submit", "documents", "steps", "contact", "immediately"]):
            score += 1
        else:
            notes.append("weak actionable steps")

    # Penalize vague phrasing
    if any(k in a for k in ["it depends", "usually", "generally", "contact your bank"]):
        score -= 1
        notes.append("generic phrasing")

    return score, notes

def compare_answers(question, base_ans, sft_ans):
    base_score, base_notes = score_answer(question, base_ans)
    sft_score, sft_notes = score_answer(question, sft_ans)

    if sft_score > base_score:
        better = "Fine-Tuned Model"
        reason = f"More domain-specific and actionable (score {sft_score} vs {base_score})."
    elif base_score > sft_score:
        better = "Base Model"
        reason = f"More complete/safe for this question (score {base_score} vs {sft_score})."
    else:
        better = "Tie"
        reason = f"Comparable quality (score {base_score} each)."

    if base_notes or sft_notes:
        base_hint = ", ".join(base_notes[:2]) if base_notes else "none"
        sft_hint = ", ".join(sft_notes[:2]) if sft_notes else "none"
        reason += f" Notes -> Base: {base_hint}; SFT: {sft_hint}."

    return better, reason, base_score, sft_score

FastLanguageModel.for_inference(model)
sft_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}

rows = []
wins = {"Fine-Tuned Model": 0, "Base Model": 0, "Tie": 0}
score_gains = []

for q in EVAL_QUESTIONS:
    better, reason, base_score, sft_score = compare_answers(q, base_answers[q], sft_answers[q])
    wins[better] += 1
    score_gains.append(sft_score - base_score)
    rows.append({
        "Question": q,
        "Base Model Answer": base_answers[q],
        "Fine-Tuned Model Answer": sft_answers[q],
        "Which Is Better?": better,
        "Reason": reason,
    })

df_sft = pd.DataFrame(rows)
display(df_sft)

# Export Step 7 table to reports/sft_model_comparison.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "sft_model_comparison.md"

lines = [
    "# Base vs SFT Comparison (Finance FAQ Assistant)",
    "",
    "This report compares responses from the base model and instruction fine-tuned (SFT) model on the same 10 domain questions.",
    "",
    "## Evaluation Criteria",
    "",
    "- Correctness",
    "- Domain accuracy",
    "- Clarity",
    "- Safety",
    "- Helpfulness",
    "- Less generic response",
    "- Better domain-specific behavior",
    "",
    "## Comparison Table",
    "",
    "| Question | Base Model Answer | Fine-Tuned Model Answer | Which Is Better? | Reason |",
    "| --- | --- | --- | --- | --- |",
]
for row in rows:
    q = " ".join(row["Question"].split()).replace("|", "\\|")
    base = " ".join(row["Base Model Answer"].split()).replace("|", "\\|")
    sft = " ".join(row["Fine-Tuned Model Answer"].split()).replace("|", "\\|")
    better = row["Which Is Better?"].replace("|", "\\|")
    reason = " ".join(row["Reason"].split()).replace("|", "\\|")
    lines.append(f"| {q} | {base} | {sft} | {better} | {reason} |")

avg_gain = sum(score_gains) / len(score_gains) if score_gains else 0.0
lines += [
    "",
    "## SFT Takeaways",
    "",
    f"- Win count -> Fine-Tuned: {wins['Fine-Tuned Model']}, Base: {wins['Base Model']}, Tie: {wins['Tie']}",
    f"- Average score gain (SFT - Base): {avg_gain:.2f}",
    "- Typical SFT improvements: better domain specificity, clearer steps, and improved finance safety tone.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 7 report written:", out_path)

## Interactive demo

In [ ]:
# Finance-only ipywidgets demo
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


# Stage 3 — Finance DPO Preference Alignment
This standalone notebook creates an SFT warm start and then performs DPO preference alignment on Finance FAQ responses. It includes SFT-vs-DPO evaluation, adapter saving, and a Finance-only ipywidgets demo. The SFT warm start is included because DPO should begin from an instruction-tuned model.

In [ ]:
# Install pinned, Colab-compatible packages (already handled at the beginning)
#!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib
print("Skipping Stage 3 package install; dependencies already installed in Cell 3.")

In [ ]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


In [ ]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


In [ ]:
# Define shared Step 10 question set and base-model outputs before any stage-3 training.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=240, temperature=0.2):
    prompt = f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in base_answers.items():
    print("\nQ:", q, "\nBase:", a)

## Build SFT warm-start dataset

In [ ]:
instruction_rows=[json.loads(x) for x in (DATA_DIR/"instruction_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def sft_text(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token}"}
sft_dataset=Dataset.from_list([sft_text(r) for r in instruction_rows])
print("SFT examples:",len(sft_dataset))

In [ ]:
from trl import SFTTrainer, SFTConfig
SFT_DIR=Path("/content/finance_adapters/stage3_sft_warm_start")
sft_args=SFTConfig(output_dir=str(SFT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=10,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
sft_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=sft_dataset,args=sft_args)
sft_trainer.train(); model.save_pretrained(str(SFT_DIR)); tokenizer.save_pretrained(str(SFT_DIR))
print("Warm-start adapter saved:",SFT_DIR)

## SFT evaluation snapshot

In [ ]:
FastLanguageModel.for_inference(model)
sft_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in sft_answers.items():
    print("\nQ:", q, "\nSFT:", a)

## Build DPO dataset

In [ ]:
preference_rows=[json.loads(x) for x in (DATA_DIR/"preference_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def dpo_row(r):
    prompt=f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['prompt'].strip()}\n\n### Assistant:\n"
    return {"prompt":prompt,"chosen":r['chosen'].strip()+tokenizer.eos_token,"rejected":r['rejected'].strip()+tokenizer.eos_token}
dpo_dataset=Dataset.from_list([dpo_row(r) for r in preference_rows])
print("Preference examples:",len(dpo_dataset)); print(dpo_dataset[0])

## Train DPO alignment

In [ ]:
from trl import DPOTrainer, DPOConfig
model.train()
DPO_DIR=Path("/content/finance_adapters/stage3_dpo")
dpo_args=DPOConfig(output_dir=str(DPO_DIR),per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=5e-6,beta=0.1,max_length=MAX_SEQ_LENGTH,max_prompt_length=512,logging_steps=5,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
dpo_trainer=DPOTrainer(model=model,ref_model=None,args=dpo_args,train_dataset=dpo_dataset,processing_class=tokenizer)
dpo_trainer.train(); model.save_pretrained(str(DPO_DIR)); tokenizer.save_pretrained(str(DPO_DIR))
print("DPO adapter saved:",DPO_DIR)

## Step 10 - Final evaluation (Base vs SFT vs DPO)

In [ ]:
import re
import pandas as pd
from pathlib import Path
from IPython.display import display

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

def score_answer(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)
    score = 0
    notes = []

    if len(words) >= 30:
        score += 2
    elif len(words) >= 18:
        score += 1
    else:
        notes.append("too brief")

    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil",
        "interest", "loan", "deposit", "bank", "account", "ifsc", "otp"
    ]
    domain_hits = sum(1 for t in domain_terms if t in a)
    if domain_hits >= 2:
        score += 2
    elif domain_hits == 1:
        score += 1
    else:
        notes.append("low domain specificity")

    if "otp" in q or "unauthorized" in q or "suspicious" in q:
        if any(k in a for k in ["never share", "fraud", "report", "block", "cybercrime", "helpline"]):
            score += 2
        else:
            notes.append("missing safety guidance")

    if any(k in q for k in ["how do i", "how can i", "what should i do"]):
        if any(k in a for k in ["log in", "visit", "submit", "documents", "steps", "contact", "immediately"]):
            score += 1
        else:
            notes.append("weak actionable steps")

    if any(k in a for k in ["it depends", "usually", "generally", "contact your bank"]):
        score -= 1
        notes.append("generic phrasing")

    return score, notes

def pick_best(question, base_ans, sft_ans, dpo_ans):
    b_score, b_notes = score_answer(question, base_ans)
    s_score, s_notes = score_answer(question, sft_ans)
    d_score, d_notes = score_answer(question, dpo_ans)

    ranked = [
        ("Base Model", b_score, b_notes),
        ("SFT Model", s_score, s_notes),
        ("DPO Model", d_score, d_notes),
    ]
    ranked.sort(key=lambda x: x[1], reverse=True)

    top_name, top_score, top_notes = ranked[0]
    second_name, second_score, _ = ranked[1]

    if top_score == second_score:
        best = "Tie"
        reason = f"Top responses are similar in quality (score {top_score})."
    else:
        best = top_name
        reason = f"{top_name} is more complete/specific (score {top_score} vs {second_score})."

    note_text = ", ".join(top_notes[:2]) if top_notes else "no major weakness detected"
    reason += f" Diagnostic notes: {note_text}."
    return best, reason, b_score, s_score, d_score

FastLanguageModel.for_inference(model)
dpo_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}

rows = []
best_counts = {"Base Model": 0, "SFT Model": 0, "DPO Model": 0, "Tie": 0}
dpo_minus_sft = []

for q in EVAL_QUESTIONS:
    best, reason, b_score, s_score, d_score = pick_best(q, base_answers[q], sft_answers[q], dpo_answers[q])
    best_counts[best] += 1
    dpo_minus_sft.append(d_score - s_score)
    rows.append({
        "Question": q,
        "Base Model Answer": base_answers[q],
        "SFT Model Answer": sft_answers[q],
        "DPO Model Answer": dpo_answers[q],
        "Best Answer": best,
        "Reason": reason,
    })

df_final = pd.DataFrame(rows)
display(df_final)

# Export Step 10 table to reports/final_evaluation.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "final_evaluation.md"

lines = [
    "# Final Evaluation (Base vs SFT vs DPO)",
    "",
    "This report compares all three stages of model quality:",
    "",
    "1. Base model",
    "2. Instruction fine-tuned (SFT) model",
    "3. DPO-aligned model",
    "",
    "## Evaluation Criteria",
    "",
    "- Correctness",
    "- Helpfulness",
    "- Domain accuracy",
    "- Safety",
    "- Tone",
    "- Clarity",
    "- Hallucination reduction",
    "- Professional response quality",
    "",
    "## Final Comparison Table",
    "",
    "| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |",
    "| --- | --- | --- | --- | --- | --- |",
]
for row in rows:
    q = " ".join(row["Question"].split()).replace("|", "\\|")
    base = " ".join(row["Base Model Answer"].split()).replace("|", "\\|")
    sft = " ".join(row["SFT Model Answer"].split()).replace("|", "\\|")
    dpo = " ".join(row["DPO Model Answer"].split()).replace("|", "\\|")
    best = row["Best Answer"].replace("|", "\\|")
    reason = " ".join(row["Reason"].split()).replace("|", "\\|")
    lines.append(f"| {q} | {base} | {sft} | {dpo} | {best} | {reason} |")

avg_gain = sum(dpo_minus_sft) / len(dpo_minus_sft) if dpo_minus_sft else 0.0
lines += [
    "",
    "## Overall Findings",
    "",
    f"- Stage with largest quality jump (by wins): DPO={best_counts['DPO Model']}, SFT={best_counts['SFT Model']}, Base={best_counts['Base Model']}, Tie={best_counts['Tie']}",
    f"- Safety and professionalism signal (avg DPO-SFT score delta): {avg_gain:.2f}",
    "- Final model limitations: Some responses may remain concise or partially generic; manual review is recommended for high-stakes finance queries.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 10 report written:", out_path)

## Interactive demo

In [ ]:
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7; color: black;'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


## Inference Script demo

In [ ]:
import argparse
import os
import sys # Import sys

import torch
from peft import PeftModel
from unsloth import FastLanguageModel # NEW IMPORT


def load_model(base_model_name: str, adapter_path: str | None = None):
    # These parameters are taken from the main notebook's global settings for consistency
    _max_seq_length = 1024
    _dtype = None  # Unsloth defaults to torch.float16 for GPUs if None
    _load_in_4bit = True

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=_max_seq_length,
        dtype=_dtype,
        load_in_4bit=_load_in_4bit,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if adapter_path and os.path.exists(adapter_path):
        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()
    return tokenizer, model


def generate_answer(tokenizer, model, question: str, max_new_tokens: int = 256) -> str:
    prompt = (
        "You are a finance FAQ assistant. Provide clear, safe, and domain-specific answers.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "Answer:" in generated:
        return generated.split("Answer:", 1)[1].strip()
    return generated.strip()


def main() -> None:
    parser = argparse.ArgumentParser(description="Run inference for finance fine-tuned assistant")
    parser.add_argument(
        "--base-model",
        default=os.getenv("BASE_MODEL", "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"),
        help="Base model name or local path",
    )
    parser.add_argument(
        "--adapter-path",
        default=os.getenv("ADAPTER_PATH", ""),
        help="Optional path to LoRA/DPO adapter",
    )
    parser.add_argument(
        "--question",
        default="How can I apply for reimbursement?",
        help="Single question to run",
    )
    parser.add_argument(
        "--interactive",
        action="store_true",
        help="Run in interactive question-answer loop",
    )

    if 'ipykernel' in sys.modules:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    adapter_path = args.adapter_path if args.adapter_path else None
    tokenizer, model = load_model(args.base_model, adapter_path)

    FastLanguageModel.for_inference(model) # NEW CALL

    if args.interactive:
        print("Finance AI Assistant is ready. Type 'exit' to quit.")
        while True:
            question = input("\nQuestion: ").strip()
            if question.lower() in {"exit", "quit"}:
                break
            print("Answer:", generate_answer(tokenizer, model, question))
    else:
        print("Answer:", generate_answer(tokenizer, model, args.question))


if __name__ == "__main__":
    main()
